In [3]:
import os
import pandas as pd
import emoji
from collections import Counter

# Leitura do dataset
df = pd.read_csv("../../../00-source/Portuguese-Hate-Speech-Dataset-master/2019-05-28_portuguese_hate_speech_binary_classification.csv")
df.head()

,text,hatespeech_comb,hatespeech_G1,annotator_G1,hatespeech_G2,annotator_G2,hatespeech_G3,annotator_G3
0,@__andrea__b \nO cara vive em outro mundo\nNão...,1,1,A,1.0,V,0,E
1,@_carmeloneto Estes incompetentes não cuidam n...,0,1,D,0.0,V,0,C
2,@_carmeloneto \nOs 'cumpanhero' quebraram toda...,0,1,A,0.0,B,0,E
3,@_GlitteryKisses é isso não conseguem pensar n...,0,0,C,0.0,V,0,D
4,@_iglira bom dia macaco branco haha,1,0,A,1.0,I,1,E


In [4]:
#Extração dos emojis dos textos
def extrair_emojis(texto):
    if pd.isna(texto):
        return []
    return [char for char in texto if char in emoji.EMOJI_DATA]

df["emojis"] = df["text"].apply(extrair_emojis)

# Identificação das mensagens de ódio
df["is_odio"] = df["hatespeech_comb"]

df_odio = df[df["is_odio"] == 1] # 1 = discurso de ódio
df_nao_odio = df[df["is_odio"] == 0] # 0 = não discurso de ódio

# Conta quantas vezes cada emoji aparece nas mensagens de ódio
emojis_odio = Counter(e for lista in df_odio["emojis"] for e in lista)

# Conta quantas vezes cada emoji aparece nas mensagens de não ódio
emojis_nao_odio = Counter(e for lista in df_nao_odio["emojis"] for e in lista)

# Junta todos os emojis encontrados nas duas categorias
todos = set(emojis_odio) | set(emojis_nao_odio)

# Montagem da tabela final
df_emojis = pd.DataFrame({
    "emoji": list(todos),
    "freq_odio": [emojis_odio.get(e, 0) for e in todos],
    "freq_nao_odio": [emojis_nao_odio.get(e, 0) for e in todos],
})

df_emojis["freq_total"] = (
    df_emojis["freq_odio"] + df_emojis["freq_nao_odio"]
)

# Normalização das frequências
total_odio = df_odio.shape[0]
total_nao_odio = df_nao_odio.shape[0]

df_emojis["freq_odio_norm"] = df_emojis["freq_odio"] / total_odio
df_emojis["freq_nao_odio_norm"] = df_emojis["freq_nao_odio"] / total_nao_odio

# Salvamento da tabela em .csv
df_emojis.to_csv(
    "twitter_emojis_fortuna2019_normalizado.csv",
    index=False
)